In [1]:
from torch import no_grad, manual_seed
import numpy as np
from PIL import Image
from tifffile import imread, imwrite, COMPRESSION
from random import seed
import matplotlib.pyplot as plt
from time import time

from vulture import CompleteUpsampler
from vulture.utils import to_numpy

from xgboost import XGBRegressor

from skimage.restoration import denoise_nl_means,  denoise_tv_chambolle, denoise_wavelet

from typing import Literal, get_args
from dataclasses import dataclass


/home/ronan/threetures/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SEED = 10672
np.random.seed(SEED)
manual_seed(SEED)
seed(SEED)


@dataclass
class XGBDenoiserParams:
    n_samples: int = 800_000
    max_depth: int = 8
    eta: float = 0.3
    lambda_: float = 1.0
    alpha: float = 0.0


# Methods = Literal["TV", "Wavelet", "NLMeans", "Vulture (LU)"]
Methods = Literal["TV", "NLMeans", "Dinoiser"]


In [3]:
UP_PATH, DENOISE_PATH, AUTOENC_PATH = "../trained_models/lu_reg_ac48.pth", "../trained_models/dvt.pth", "../trained_models/dac_dv2_denoised_e500.pth"
upsampler = CompleteUpsampler("LOFTUP_COMPRESSED", UP_PATH, DENOISE_PATH, AUTOENC_PATH, device="cuda:0", to_half=True, to_eval=True, add_flash_attn=True)

In [4]:
def _get_train(img_arr: np.ndarray, feats: np.ndarray, n_samples: int) -> tuple[np.ndarray, np.ndarray]:
    h, w, c = feats.shape
    feats_flat = feats.reshape((h * w, c))
    img_flat = img_arr.reshape(h * w)

    inds = np.arange(0, len(feats_flat))
    np.random.shuffle(inds)
    subsample_inds = inds[:n_samples]

    train = feats_flat[subsample_inds]
    targ = img_flat[subsample_inds] / 255.0

    return train, targ

def our_denosing(img_arr: np.ndarray, upsampler: CompleteUpsampler, params: XGBDenoiserParams) -> np.ndarray:
    assert img_arr.ndim == 2, "Input image must be 2D"
    with no_grad():
        hr_feats = upsampler.forward(img_arr)
    hr_feats_np = to_numpy(hr_feats).astype(np.float32).transpose((1, 2, 0))
    h, w, c = hr_feats_np.shape
    hr_feats_flat = hr_feats_np.reshape((h * w, c))

    train, targ = _get_train(img_arr, hr_feats_np, n_samples=800_000)

    denoiser = XGBRegressor(**params.__dict__)
    denoiser.fit(train, targ)
    pred = denoiser.predict(hr_feats_flat)
    pred = np.clip(pred, 0, 1) * 255.0
    return pred.reshape(h, w).astype(np.uint8)

def chambolle_denoising(img_arr: np.ndarray, weight: float = 0.1) -> np.ndarray:
    img_arr_rgb = np.stack([img_arr] * 3, axis=-1)
    return denoise_tv_chambolle(img_arr_rgb, weight=weight)

def wavelet_denoising(img_arr: np.ndarray, method: str = 'BayesShrink', mode: str = 'soft') -> np.ndarray:
    img_arr_rgb = np.stack([img_arr] * 3, axis=-1)
    wavelet = denoise_wavelet(img_arr_rgb, channel_axis=-1, convert2ycbcr=True, rescale_sigma=True)
    return wavelet

def nlm_denoising(img_arr: np.ndarray, patch_size: int = 7, patch_distance: int = 11, h: float = 0.1) -> np.ndarray:
    return denoise_nl_means(img_arr, patch_size=patch_size, patch_distance=patch_distance, h=h, fast_mode=True, channel_axis=None)

In [14]:
def rescale_to_255(img_arr: np.ndarray) -> np.ndarray:
    img = img_arr.astype(np.float32)
    amin, amax = img.min(), img.max()
    img = (img - amin) / (amax - amin)
    return (img * 255).astype(np.uint8)

In [13]:
def get_image(fname: str) -> np.ndarray:
    if ".tif" in fname or ".tiff" in fname:
        img = imread(fname)
        if img.ndim == 3:
            img = img[:, :, 0]
        return rescale_to_255(img)

    img = Image.open(fname).convert("L")
    img_arr = np.array(img, dtype=np.uint8)
    return img_arr

In [6]:
methods: list[Methods] = get_args(Methods)
img_fnames = ("sofc_crop.tif", "bat2_crop.tif", "tem_crop.tif")
path = "fig_data/sem_denoising"
imgs = [get_image(f"{path}/{fname}") for fname in img_fnames]

outputs: dict[Methods, list[np.ndarray]] = {method: [] for method in methods}
times: dict[Methods, list[float]] = {method: [] for method in methods}

for img in imgs:
    for method in methods:
        start_time = time()
        if method == "TV":
            denoised_img = chambolle_denoising(img)
        elif method == "Wavelet":
            denoised_img = wavelet_denoising(img)
        elif method == "NLMeans":
            denoised_img = nlm_denoising(img)
        elif method == "Dinoiser":
            params = XGBDenoiserParams()
            denoised_img = our_denosing(img, upsampler, params)
        else:
            raise ValueError(f"Unknown denoising method: {method}")
        
        elapsed_time = time() - start_time
        outputs[method].append(denoised_img)
        times[method].append(elapsed_time)


In [19]:
def add_inset_zoom(
    ax,
    xywh: list[int],
    fig_xywh: list[float],
    img_arr: np.ndarray,
) -> object:
    x0, y0, w, h = xywh
    inset_data = np.zeros_like(img_arr)
    inset_data[y0 : y0 + h, x0 : x0 + w, :] = img_arr[y0 : y0 + h, x0 : x0 + w, :]

    axin = ax.inset_axes(fig_xywh, xlim=(x0, x0 + w), ylim=(y0, y0 + h))
    axin.set_xticks([])
    axin.set_yticks([])

    axin.imshow(inset_data, cmap="binary_r", interpolation='nearest', vmin=0, vmax=255)
    
    # Disable clipping on the indicator box and connector lines
    indicator_box, connector_lines = ax.indicate_inset_zoom(axin, edgecolor="black", lw=1)
    indicator_box.set_clip_on(False)
    for line in connector_lines:
        line.set_clip_on(False)

    axin.set_ylim((y0 + h, y0))
    axin.patch.set_edgecolor("black")
    axin.patch.set_linewidth(1)
    return axin

In [20]:



def hide_axes(ax):
    ax.set_xticks([])
    ax.set_yticks([])

plt.style.use("thesis.mplstyle")
fig, axs = plt.subplots(len(imgs), 1 + len(methods), figsize=(7, 2.3 * 2.5))


inset_zooms_rel_xyhw = [(0.6, 0.3, 0.3, 0.3), (0.4, 0.25, 0.3, 0.3), (0.5, 0.2, 0.3, 0.3), (0.0, 0.3, 0.3, 0.3)]
inset_zoom_fig_locs = [(0.70, 0.1, 0.6, 0.6) for _ in range(len(imgs))] 

map_keys = lambda x: rf"\textbf{{{x}}}" if 'Dinoiser' in x else x   

for i, img in enumerate(imgs):
    h, w = img.shape
    
    # Force highest zorder on the first column
    axs[i, 0].set_zorder(100) 
    
    axs[i, 0].imshow(img, cmap='gray', vmin=0, vmax=255)
    if i == 0:
        axs[i, 0].set_title("Original")
    axs[i, 0].set_ylabel(f"({h}x{w})")
    hide_axes(axs[i, 0])

    scales = (w, h, w, h)
    inset_zoom_abs_xyhw = list(int(scale * rel) for scale, rel in zip(scales, inset_zooms_rel_xyhw[i]))

    img_arr_rgb = np.stack([img] * 3, axis=-1)
    add_inset_zoom(axs[i, 0], inset_zoom_abs_xyhw, inset_zoom_fig_locs[i], img_arr_rgb)

    for j, method in enumerate(methods):
        col_idx = j + 1
        
        # Strictly decrease zorder for each subsequent column
        axs[i, col_idx].set_zorder(100 - col_idx)
        
        denoised_img = outputs[method][i]
        denoised_img = rescale_to_255(denoised_img)
        axs[i, col_idx].imshow(denoised_img, cmap='gray', vmin=0, vmax=255, interpolation='nearest')
        if i == 0:
            title = map_keys(method)
            axs[i, col_idx].set_title(title)
        axs[i, col_idx].set_xlabel(f"Time: {times[method][i]:.2f}s")
        hide_axes(axs[i, col_idx])

        if denoised_img.ndim == 2:
            denoised_img_rgb = np.stack([denoised_img] * 3, axis=-1)
        else:
            denoised_img_rgb = denoised_img

        add_inset_zoom(axs[i, col_idx], inset_zoom_abs_xyhw, inset_zoom_fig_locs[i], denoised_img_rgb)
SAVE = True
if SAVE:
    plt.savefig("fig_pdf/sem_denoising.pdf", dpi=300,)
    plt.close()